# Chest X-ray CNN: dropout, weight decay and a hyperparameter grid search

The small CNN from `04_chest_Xray_CNN.ipynb`, with these changes:

* **Dropout** between the fully connected layers, and a **ReLU** between them. Without the ReLU, the three `Linear` layers of notebook 04 collapse into a single linear map.
* **Weight decay** (L2 regularisation) set through the optimizer.
* **Pixels scaled to [0, 1]**. Notebook 04 feeds raw 0–255 values, which makes SGD unstable at usual learning rates.
* A **grid search** over dropout, optimizer type, learning rate, weight decay and optimizer-specific settings (Adam `beta1`, SGD `momentum`).

**Honest model selection.** The grid is scored on a validation split carved out of `train/` (stratified 80/20). The `test/` folder is used **once**, for the chosen model only. Notebook 04 used `test/` for early stopping, so its accuracy is optimistic. The accuracy here is not.

Outputs: `cnn_gridsearch_results.csv`, `cnn_regularized_model.pt`, `cnn_regularized_model.json`.

In [ ]:
## Colab only: get the code and data (uncomment)
# !git clone https://github.com/wxchew/xray_project.git
# %cd xray_project

In [ ]:
import copy
import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

import torchvision
from torchvision.io import read_image
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import roc_auc_score, confusion_matrix

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
%%time
## Decoding JPEGs every epoch would dominate a grid search, so every image is decoded once
## and kept in memory as uint8 (~300 MB for all 224x224 images).
TRAIN_DIR = 'data/chest_xray_224/train'
TEST_DIR = 'data/chest_xray_224/test'

def load_folder(path):
    ds = torchvision.datasets.ImageFolder(path, loader = read_image)
    X = torch.stack([x for x, _ in ds])                              # N,1,224,224 uint8
    y = torch.tensor(ds.targets, dtype = torch.float32).unsqueeze(1)  # N,1
    return X, y, ds.class_to_idx

X_all, y_all, class_to_idx = load_folder(TRAIN_DIR)
X_test, y_test, test_class_to_idx = load_folder(TEST_DIR)
assert class_to_idx == test_class_to_idx == {'NORMAL': 0, 'PNEUMONIA': 1}

VAL_FRACTION = 0.2
idx_train, idx_val = train_test_split(np.arange(len(y_all)),
                                      test_size = VAL_FRACTION,
                                      stratify = y_all.squeeze().numpy(),
                                      random_state = SEED)
X_train, y_train = X_all[idx_train], y_all[idx_train]
X_val, y_val = X_all[idx_val], y_all[idx_val]

for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f"{name:5s} n={len(y):5d}  pneumonia fraction={y.mean().item():.3f}")

In [ ]:
class RegCNN(nn.Module):
    def __init__(self, dropout = 0.0, input_size = 224):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 2, kernel_size = 16, stride = 4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
            nn.Conv2d(2, 4, kernel_size = 5, stride = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2))
        self.flatten = nn.Flatten()
        with torch.no_grad():
            flatten_output_dim = self.flatten(self.conv(torch.zeros(1, 1, input_size, input_size))).shape[1]
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(flatten_output_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(8, 1),
            nn.Sigmoid())

    def forward(self, x):
        return self.classifier(self.flatten(self.conv(x)))


RegCNN(dropout = 0.5)

In [ ]:
def to_input(X):
    return X.to(device).float() / 255.0


def make_optimizer(cfg, params):
    if cfg['optimizer'] == 'adam':
        # AdamW = Adam with decoupled weight decay (plain Adam mixes L2 into its adaptive step)
        return torch.optim.AdamW(params, lr = cfg['lr'], weight_decay = cfg['weight_decay'],
                                 betas = (cfg['beta1'], 0.999))
    if cfg['optimizer'] == 'sgd':
        return torch.optim.SGD(params, lr = cfg['lr'], weight_decay = cfg['weight_decay'],
                               momentum = cfg['momentum'], nesterov = True)
    raise ValueError(cfg['optimizer'])


def evaluate(model, X, y, batch_size = 256):
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            probs.append(model(to_input(X[i:i + batch_size])).cpu())
    probs = torch.cat(probs)
    if not torch.isfinite(probs).all():
        return np.nan, np.nan, probs.squeeze(1).numpy()
    loss =nn.functional.binary_cross_entropy(probs, y).item()
    accuracy = ((probs > 0.5).float() == y).float().mean().item()
    return loss, accuracy, probs.squeeze(1).numpy()


def fit(cfg, max_epochs, patience, batch_size = 64, echo = False):
    torch.manual_seed(SEED)
    model = RegCNN(dropout = cfg['dropout']).to(device)
    optimizer = make_optimizer(cfg, model.parameters())
    loss_fn = nn.BCELoss()
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size = batch_size, shuffle = True,
                        generator = torch.Generator().manual_seed(SEED))

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best = {'val_loss': np.inf, 'epoch': 0, 'state': None}
    wait = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        total = 0.0
        for X, y in loader:
            optimizer.zero_grad()
            pred = model(to_input(X))
            if not torch.isfinite(pred).all():
                total = np.nan  # diverged; BCELoss would raise on NaN input
                break
            loss = loss_fn(pred, y.to(device))
            loss.backward()
            optimizer.step()
            total += loss.item() * len(X)
        train_loss = total / len(X_train)
        if not np.isfinite(train_loss):
            break
        val_loss, val_acc, _ = evaluate(model, X_val, y_val)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if echo:
            print(f"epoch {epoch:3d}  train {train_loss:.4f}  val {val_loss:.4f}  val acc {val_acc:.3f}")
        if val_loss < best['val_loss'] - 1e-4:
            best = {'val_loss': val_loss, 'val_acc': val_acc, 'epoch': epoch,
                    'state': copy.deepcopy(model.state_dict())}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    return best, history

In [ ]:
## optimizer-specific settings only appear in their own sub-grid, so no config is a duplicate
common = {'dropout': [0.0, 0.25, 0.5],
          'weight_decay': [0.0, 1e-4, 1e-3]}
param_grid = [
    {**common, 'optimizer': ['adam'], 'lr': [1e-4, 1e-3], 'beta1': [0.9, 0.8]},
    {**common, 'optimizer': ['sgd'],  'lr': [1e-3, 1e-2], 'momentum': [0.8, 0.9]},
]
configs = list(ParameterGrid(param_grid))

MAX_EPOCHS = 40
PATIENCE = 6
BATCH_SIZE = 64
print(len(configs), 'configurations, at most', MAX_EPOCHS, 'epochs each')

In [ ]:
%%time
rows, histories, states = [], {}, {}
for i, cfg in enumerate(configs):
    t0 = time.time()
    best, history = fit(cfg, MAX_EPOCHS, PATIENCE, BATCH_SIZE)
    rows.append({'config_id': i, **cfg,
                 'best_epoch': best['epoch'],
                 'epochs_run': len(history['val_loss']),
                 'val_loss': best['val_loss'],
                 'val_acc': best.get('val_acc', np.nan),
                 'seconds': round(time.time() - t0, 1)})
    histories[i] = history
    states[i] = best['state']
    print(f"[{i + 1:2d}/{len(configs)}] {cfg}  ->  val loss {best['val_loss']:.4f}  "
          f"val acc {best.get('val_acc', np.nan):.3f}  (best epoch {best['epoch']})")

results = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop = True)
results.to_csv('cnn_gridsearch_results.csv', index = False)
results.head(10)

In [ ]:
## average validation loss per value of each hyperparameter (marginal effects)
for col in ['dropout', 'weight_decay', 'optimizer', 'lr']:
    display(results.groupby(col)[['val_loss', 'val_acc']].agg(['mean', 'min']).round(4))

best_row = results.iloc[0]
best_id = int(best_row['config_id'])
best_cfg = configs[best_id]
print('best configuration:', best_cfg)

h = histories[best_id]
plt.plot(range(1, len(h['train_loss']) + 1), h['train_loss'], label = 'train')
plt.plot(range(1, len(h['val_loss']) + 1), h['val_loss'], label = 'validation')
plt.axvline(best_row['best_epoch'], linestyle = '--', color = 'r', label = 'best epoch')
plt.xlabel('epoch')
plt.ylabel('BCE loss')
plt.title('best configuration')
plt.legend()
plt.show()

In [ ]:
## the test set is used once, for the selected model only
model = RegCNN(dropout = best_cfg['dropout']).to(device)
model.load_state_dict(states[best_id])

train_loss, train_acc, _ = evaluate(model, X_train, y_train)
val_loss, val_acc, _ = evaluate(model, X_val, y_val)
test_loss, test_acc, test_probs = evaluate(model, X_test, y_test)
y_true = y_test.squeeze(1).numpy()
test_auc = roc_auc_score(y_true, test_probs)
tn, fp, fn, tp = confusion_matrix(y_true, test_probs > 0.5, labels = [0, 1]).ravel()

print(f"train acc {train_acc:.3f} | val acc {val_acc:.3f} | test acc {test_acc:.3f} | test ROC-AUC {test_auc:.3f}")
print(f"test sensitivity {tp / (tp + fn):.3f} | test specificity {tn / (tn + fp):.3f}")
print(pd.DataFrame([[tn, fp], [fn, tp]], index = ['true NORMAL', 'true PNEUMONIA'],
                   columns = ['pred NORMAL', 'pred PNEUMONIA']))

In [ ]:
MODEL_PATH = 'cnn_regularized_model.pt'
META_PATH = 'cnn_regularized_model.json'

native = lambda v: v.item() if hasattr(v, 'item') else v
metadata = {
    'model': {
        'class': 'RegCNN (defined in 04c_chest_Xray_CNN_gridsearch.ipynb)',
        'dropout': best_cfg['dropout'],
        'output': 'probability of PNEUMONIA',
        'decision_threshold': 0.5,
        'parameters': sum(p.numel() for p in model.parameters()),
    },
    'input': {'shape': [1, 224, 224], 'color': 'grayscale', 'scaling': 'uint8 / 255'},
    'data': {
        'class_to_idx': class_to_idx,
        'train_dir': TRAIN_DIR, 'test_dir': TEST_DIR,
        'split': f'stratified {1 - VAL_FRACTION:.0%}/{VAL_FRACTION:.0%} of train_dir, random_state={SEED}',
        'n_train': len(y_train), 'n_val': len(y_val), 'n_test': len(y_test),
        'note': 'grid search and early stopping used the validation split only; test used once',
    },
    'selection': {
        'criterion': 'lowest validation BCE loss',
        'n_configs': len(configs),
        'param_grid': param_grid,
        'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'batch_size': BATCH_SIZE,
        'results_csv': 'cnn_gridsearch_results.csv',
    },
    'training': {
        'config': {k: native(v) for k, v in best_cfg.items()},
        'optimizer_class': 'AdamW' if best_cfg['optimizer'] == 'adam' else 'SGD(nesterov=True)',
        'best_epoch': int(best_row['best_epoch']),
        'epochs_run': int(best_row['epochs_run']),
        'train_losses': histories[best_id]['train_loss'],
        'val_losses': histories[best_id]['val_loss'],
        'seed': SEED,
    },
    'metrics': {
        'train_accuracy': train_acc, 'val_accuracy': val_acc, 'val_loss': val_loss,
        'test_accuracy': test_acc, 'test_loss': test_loss, 'test_roc_auc': float(test_auc),
        'test_confusion': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    },
    'environment': {
        'torch': torch.__version__, 'torchvision': torchvision.__version__, 'device': device,
        'saved_utc': datetime.now(timezone.utc).isoformat(timespec = 'seconds'),
    },
}

torch.save({'model_state_dict': model.state_dict(),
            'config': best_cfg,
            'train_accuracy': train_acc,
            'valid_accuracy': val_acc,
            'test_accuracy': test_acc,
            'metadata': metadata},
           MODEL_PATH)
with open(META_PATH, 'w') as fh:
    json.dump(metadata, fh, indent = 2)
print('saved', MODEL_PATH, 'and', META_PATH)

In [ ]:
## reload check
ckpt = torch.load(MODEL_PATH, map_location = 'cpu', weights_only = False)
reloaded = RegCNN(dropout = ckpt['config']['dropout']).to(device)
reloaded.load_state_dict(ckpt['model_state_dict'], strict = True)
_, reloaded_acc, _ = evaluate(reloaded, X_test, y_test)
assert np.isclose(reloaded_acc, ckpt['test_accuracy']), (reloaded_acc, ckpt['test_accuracy'])
print('reloaded test accuracy:', reloaded_acc)